# 🫁 LungRay Multi-Task Baseline Model Training (Phase 3 / Gate 2)
## Multi-Task 3-Head CADx Architecture with Shared DenseNet-121 Encoder
---
| Thuộc tính | Giá trị |
|:---|:---|
| **Nền tảng** | **LungRay** (Chest X-ray Multi-Task CADx + WSOL Grounding Engine) |
| **Giai đoạn / Gate** | **Phase 3 / Delivery Gate 2 (Tuần 5–6)** |
| **Kiến trúc Backbone** | Shared DenseNet-121 (CheXNet baseline) |
| **Chuẩn kích thước ảnh** | **$512 	imes 512$ Standard Medical Resolution** đồng nhất cho cả 3 heads |
| **Chiến lược lấy mẫu** | `WeightedMultiTaskBatchSampler` cố định **60% NIH : 25% COVID : 15% TB** |
| **Cơ chế đường dẫn** | **Smart Recursive Path Indexer ($O(1)$ lookup)** chống lỗi thư mục lồng nhau |
| **Môi trường phần cứng** | Kaggle GPU (NVIDIA P100 / 2x T4) / Local CPU Dry-Run |



### 1. Khởi Tạo Môi Trường & Kiểm Tra Tăng Tốc Phần Cứng (Hardware Acceleration Discovery)
Tự động nhận diện thiết bị tính toán (Kaggle GPU P100 / T4 hoặc CPU Fallback), cấu hình Mixed Precision (AMP FP16) và cố định hạt giống ngẫu nhiên (`seed=42`).


In [ ]:
# --- 0. TỰ ĐỘNG ĐỒNG BỘ MÃ NGUỒN LUNGRAY TRÊN KAGGLE CLOUD ---
import os
import sys

kaggle_base = '/kaggle/input'
if os.path.exists(kaggle_base):
    repo_dir = '/kaggle/working/LungRay'
    if not os.path.exists(repo_dir):
        print('[*] Phát hiện môi trường Kaggle Cloud -> Đang tự động clone repository LungRay...')
        !git clone https://github.com/philipsgn/LungRay.git {repo_dir}
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    print(f'[OK] Đã kết nối mã nguồn LungRay ({repo_dir}) vào môi trường Kaggle!')

import os
import sys
import math
import time
import random
from typing import Dict, List, Tuple, Any

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

# Fix random seed for strict reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] Compute Device: {device}')
if torch.cuda.is_available():
    print(f'[*] GPU Accelerator: {torch.cuda.get_device_name(0)}')
    print(f'[*] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('[!] Running on CPU fallback. Luu y: Huan luyen 25 epochs can chay tren Kaggle GPU (P100 / T4).')



### 2. Dataset Structure & Resolution Inspector (Kiểm Toán Cấu Trúc & Kích Thước Ảnh)
Tự động quét cấu trúc cây thư mục trên `/kaggle/input/` (hoặc `dataset/`), thiết lập **Smart Recursive Path Indexer** để lập bản đồ $O(1)$ cho 100% file ảnh bất kể nằm sâu bao nhiêu cấp thư mục con.


### 🔍 CELL ĐỘC LẬP: KAGGLE RESOLUTION PROFILER (KIỂM TRA ĐỘ PHÂN GIẢI THỰC TẾ)
Chạy cell này đầu tiên trên Kaggle để quét 100 ảnh ngẫu nhiên từ mỗi bộ dataset mounted:
- Thống kê **Min, Max, Median (Width x Height)** và số kênh màu (Grayscale/RGB).
- Kiểm tra xem có ảnh nào nhỏ hơn 512x512 hay không (để tránh tuyệt đối việc upsample làm mờ ảnh).


In [ ]:
import os
import random
from PIL import Image
import pandas as pd

kaggle_base = '/kaggle/input'
search_base = kaggle_base if os.path.exists(kaggle_base) else ('../dataset' if os.path.exists('../dataset') else 'dataset')

def profile_dataset_resolutions(root_dir, sample_size=100):
    image_files = []
    for r, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_files.append(os.path.join(r, f))
    
    if not image_files:
        return {'Total_Images': 0, 'Samples_Checked': 0, 'Resolution_Min': 'N/A', 'Resolution_Max': 'N/A', 'Resolution_Median': 'N/A', 'Has_Small_Images (<512)': 'N/A'}
    
    sampled = random.sample(image_files, min(sample_size, len(image_files)))
    widths, heights, modes = [], [], []
    for p in sampled:
        try:
            with Image.open(p) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
                modes.append(img.mode)
        except Exception:
            pass
            
    has_small = any(w < 512 or h < 512 for w, h in zip(widths, heights))
    most_common_w = pd.Series(widths).mode()[0] if widths else 0
    most_common_h = pd.Series(heights).mode()[0] if heights else 0
    
    return {
        'Total_Images': len(image_files),
        'Samples_Checked': len(sampled),
        'Resolution_Min': f'{min(widths)}x{min(heights)}',
        'Resolution_Max': f'{max(widths)}x{max(heights)}',
        'Resolution_Median': f'{int(pd.Series(widths).median())}x{int(pd.Series(heights).median())}',
        'Resolution_Mode': f'{most_common_w}x{most_common_h}',
        'Has_Small (<512)': 'CÓ (<512) ⚠️' if has_small else 'KHÔNG (>=512) ✅',
        'Color_Modes': list(set(modes))
    }

print(f'[*] Dang quet phan giai tu goc: {search_base} ...')
results = []

# Tim kiem cac thu muc con cua 3 dataset
subdirs = [os.path.join(search_base, d) for d in os.listdir(search_base) if os.path.isdir(os.path.join(search_base, d))]
for d in subdirs:
    d_name = os.path.basename(d)
    prof = profile_dataset_resolutions(d, sample_size=100)
    prof['Dataset_Folder'] = d_name
    results.append(prof)

df_prof = pd.DataFrame(results)
cols = ['Dataset_Folder', 'Total_Images', 'Resolution_Median', 'Resolution_Min', 'Resolution_Max', 'Has_Small (<512)', 'Color_Modes']
df_prof = df_prof[[c for c in cols if c in df_prof.columns]]
print('\n=== BÁO CÁO ĐỘ PHÂN GIẢI THỰC TẾ TRÊN KAGGLE ===')
print(df_prof.to_string(index=False))


In [ ]:
# --- 2.1. Quét Cây Thư Mục & Tự Động Định Danh Dataset ---
kaggle_base = '/kaggle/input'
is_kaggle = os.path.exists(kaggle_base)

def scan_dir_tree(base_path, max_depth=2, current_depth=0):
    if not os.path.exists(base_path) or current_depth > max_depth:
        return
    indent = '    ' * current_depth
    try:
        entries = os.listdir(base_path)
    except Exception as e:
        return
    dirs = [d for d in entries if os.path.isdir(os.path.join(base_path, d))]
    files = [f for f in entries if os.path.isfile(os.path.join(base_path, f))]
    img_count = sum(1 for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg')))
    
    print(f'{indent}📁 {os.path.basename(base_path)}/ (dirs: {len(dirs)}, images: {img_count})')
    for d in dirs[:5]:  # Hien thi toi da 5 thu muc con de giu output gon gang
        scan_dir_tree(os.path.join(base_path, d), max_depth, current_depth + 1)
    if len(dirs) > 5:
        print(f'{indent}    ... ({len(dirs) - 5} thu muc con khac)')

print('=== CẤU TRÚC THƯ MỤC TRÊN HỆ THỐNG ===')
if is_kaggle:
    print('Scan tai /kaggle/input/:')
    for item in os.listdir(kaggle_base):
        scan_dir_tree(os.path.join(kaggle_base, item), max_depth=2)
else:
    prefix = '../' if os.path.exists('../dataset') else ''
    scan_dir_tree(os.path.join(prefix, 'dataset'), max_depth=2)



In [ ]:
# --- 2.2. Smart Recursive Path Indexer (Lập Bản Đồ Đường Dẫn Tuyệt Đối O(1)) ---
def build_recursive_file_map(root_dir: str) -> Dict[str, str]:
    '''Quet de quy thu thap moi file anh .png, .jpg, .jpeg thanh dictionary filename -> full_path (Loai bo 100% masks).'''
    file_map = {}
    if not os.path.exists(root_dir):
        return file_map
    for root, dirs, files in os.walk(root_dir):
        # STRICT RULE: Skip any mask directories (COVID dataset contains parallel masks/ folders with identical filenames)
        dirs[:] = [d for d in dirs if d.lower() != 'masks']
        if 'masks' in os.path.normpath(root).lower().split(os.sep):
            continue
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.dcm')):
                file_map[f] = os.path.join(root, f)
    return file_map

# Xac dinh thu muc goc cua 3 bo du lieu (Ho tro nested directories tren Kaggle)
if is_kaggle:
    def find_kaggle_root(target_keys):
        for root, dirs, files in os.walk(kaggle_base):
            bname = os.path.basename(root).lower()
            if any(k in bname for k in target_keys):
                return root
            if any(k in [d.lower() for d in dirs] for k in target_keys):
                return root
        return None

    nih_dir = find_kaggle_root(['lungray-nih-512', 'nih-512', 'nih', 'chestxray', 'chest-xrays']) or 'dataset/nih_chestxray14'
    covid_dir = find_kaggle_root(['lungray-covidx-512', 'covidx-512', 'covidx-cxr2', 'covidx', 'covid-19_radiography_dataset', 'covid19-radiography-database', 'covid']) or 'dataset/covidx_cxr4'
    tb_dir = find_kaggle_root(['tb_chest_radiography_database', 'tuberculosis-tb-chest-xray-dataset', 'tuberculosis']) or 'dataset/tb_chestxray'
    split_dir = 'dataset/splits' if os.path.exists('dataset/splits') else os.path.join(kaggle_base, 'lungray-splits')
    output_dir = '/kaggle/working'
else:
    prefix = '../' if os.path.exists('../dataset') else ''
    nih_dir = os.path.join(prefix, 'dataset/nih_chestxray14')
    covid_dir = os.path.join(prefix, 'dataset/covidx_cxr4') if os.path.exists(os.path.join(prefix, 'dataset/covidx_cxr4')) else os.path.join(prefix, 'dataset/covid19_radiography')
    tb_dir = os.path.join(prefix, 'dataset/tb_chestxray')
    split_dir = os.path.join(prefix, 'dataset/splits')
    output_dir = os.path.join(prefix, 'checkpoints')

os.makedirs(output_dir, exist_ok=True)
results_dir = 'results/baseline' if not is_kaggle else '/kaggle/working/results'
os.makedirs(results_dir, exist_ok=True)

print('\n[*] Dang lap ban do de quy cho 3 bo du lieu...')
start_idx = time.time()
file_map_nih = build_recursive_file_map(nih_dir)
file_map_covid = build_recursive_file_map(covid_dir)
file_map_tb = build_recursive_file_map(tb_dir)
print(f'[OK] Hoan tat lap chi muc trong {time.time() - start_idx:.2f}s:')
print(f' - NIH File Map   : {len(file_map_nih):,d} anh')
print(f' - COVID File Map : {len(file_map_covid):,d} anh')
print(f' - TB File Map    : {len(file_map_tb):,d} anh')



### 3. Resolution Audit & On-the-Fly Standardization Proof (Kiểm Toán Kích Thước Ảnh)
Khảo sát kích thước gốc thực tế của 3 bộ dữ liệu:
- **NIH ChestX-ray14**: Kích thước gốc $1024 	imes 1024$.
- **COVID-19 Radiography**: Kích thước gốc $299 	imes 299$.
- **Tuberculosis (TB)**: Kích thước gốc $512 	imes 512$.
Chứng minh cơ chế **On-the-Fly Standardization** đưa 100% ảnh về tensor đồng nhất $(3, 512, 512)$ mà không tốn dung lượng ổ đĩa.


In [ ]:
# Ensure src package is importable
import sys
for p in ['.', '..']:
    if os.path.exists(os.path.join(p, 'src')):
        if p not in sys.path:
            sys.path.insert(0, p)
        break

from src.data.transforms import get_train_transforms, get_val_transforms
val_tf = get_val_transforms(img_size=512)

def audit_dataset_resolution(name, file_map, n_samples=3):
    print(f'\n--- KIỂM TOÁN KÍCH THƯỚC: {name} ---')
    if not file_map:
        print(f'[!] File map trong, bo qua audit truc tiep (se dung fallback).')
        return
    sample_files = random.sample(list(file_map.keys()), min(n_samples, len(file_map)))
    for fname in sample_files:
        fpath = file_map[fname]
        img = cv2.imread(fpath)
        if img is not None:
            raw_h, raw_w, raw_c = img.shape
            # Dua qua transforms chuan hoa
            transformed = val_tf(image=img)
            t_tensor = transformed['image']
            print(f'  Anh: {fname:<25} | Kich thuoc goc: {raw_w}x{raw_h} (channels: {raw_c}) -> Sau chuan hoa: {list(t_tensor.shape)}')
        else:
            print(f'  Anh: {fname:<25} | Khong doc duoc qua cv2.')

audit_dataset_resolution('NIH ChestX-ray14 (Head A)', file_map_nih, n_samples=3)
audit_dataset_resolution('Tuberculosis Database (Head B)', file_map_tb, n_samples=3)
audit_dataset_resolution('COVID-19 Radiography (Head C)', file_map_covid, n_samples=3)

print('\n[CHỨNG MINH KỸ THUẬT] Du kich thuoc anh goc la 1024x1024 (NIH), 299x299 (COVID) hay 512x512 (TB):')
print('-> Pipeline On-the-Fly luon dam bao 100% anh di vao mang hoc sau deu co shape chuan: (3, 512, 512).')



### 4. Nạp Chỉ Mục Phân Chia & DataLoaders Đa Nhiệm (Zero Data Leakage)
Đọc 3 file split CSV đã kiểm định (`patient_splits.csv`, `covid_splits.csv`, `tb_splits.csv`), tích hợp `file_map` vào DataLoader và cấu hình `WeightedMultiTaskBatchSampler` cố định tỷ lệ 60:25:15.


In [ ]:
from src.data.dataset import ChestXrayDataset
from src.data.sampler import WeightedMultiTaskBatchSampler
from src.models.trainer import multitask_collate_fn

def locate_split_file(filename, candidates):
    for c in candidates:
        if not c or not os.path.exists(c): continue
        direct = os.path.join(c, filename)
        if os.path.exists(direct): return direct
        for root, _, files in os.walk(c):
            if filename in files: return os.path.join(root, filename)
    return None

split_cands = ['/kaggle/working/LungRay/dataset/splits', '/kaggle/working/LungRay/data/splits', split_dir, 'dataset/splits', '../dataset/splits', nih_dir, covid_dir, tb_dir, '/kaggle/input/lungray-splits']
nih_split_path = locate_split_file('patient_splits.csv', split_cands)
covid_split_path = locate_split_file('covid_splits.csv', split_cands)
tb_split_path = locate_split_file('tb_splits.csv', split_cands)

assert nih_split_path and os.path.exists(nih_split_path), f'Missing patient_splits.csv in {split_cands}'
assert covid_split_path and os.path.exists(covid_split_path), f'Missing covid_splits.csv in {split_cands}'
assert tb_split_path and os.path.exists(tb_split_path), f'Missing tb_splits.csv in {split_cands}'
print(f'[OK] Resiliently Found Splits:\n - NIH: {nih_split_path}\n - COVID: {covid_split_path}\n - TB: {tb_split_path}')

df_nih = pd.read_csv(nih_split_path)
df_covid = pd.read_csv(covid_split_path)
df_tb = pd.read_csv(tb_split_path)

print('=== KIỂM TOÁN ĐỘ TRÙNG KHỚP CỦA SPLIT VỚI ĐĨA ===')
def check_coverage(df, img_col, file_map, name):
    if not file_map:
        print(f'[{name}] File map trong (fallback mode).')
        return
    matched = sum(1 for img in df[img_col] if str(img) in file_map)
    total = len(df)
    pct = (matched / total) * 100.0 if total > 0 else 0
    print(f'[{name}] Khop: {matched:,d} / {total:,d} anh ({pct:.2f}%)')

check_coverage(df_nih, 'Image Index', file_map_nih, 'NIH ChestX-ray14')
check_coverage(df_covid, 'Image Index', file_map_covid, 'COVID-19 Radiography')
check_coverage(df_tb, 'Image Index', file_map_tb, 'Tuberculosis Database')

train_tf = get_train_transforms(img_size=512)

# Build Datasets (Truyen file_map de co O(1) path lookup toc do cao)
train_nih_df = df_nih[df_nih.split == 'train'].reset_index(drop=True)
val_nih_df = df_nih[df_nih.split == 'val'].reset_index(drop=True)
test_nih_df = df_nih[df_nih.split == 'test'].reset_index(drop=True)

train_covid_df = df_covid[df_covid.split == 'train'].reset_index(drop=True)
val_covid_df = df_covid[df_covid.split == 'val'].reset_index(drop=True)
test_covid_df = df_covid[df_covid.split == 'test'].reset_index(drop=True)

train_tb_df = df_tb[df_tb.split == 'train'].reset_index(drop=True)
val_tb_df = df_tb[df_tb.split == 'val'].reset_index(drop=True)
test_tb_df = df_tb[df_tb.split == 'test'].reset_index(drop=True)

pre_resized_nih = '512' in os.path.basename(nih_dir).lower() or '512' in str(nih_dir).lower()
pre_resized_covid = '512' in os.path.basename(covid_dir).lower() or '512' in str(covid_dir).lower()
print(f'[*] Dual-Mode Resolution: NIH pre_resized={pre_resized_nih}, COVIDx pre_resized={pre_resized_covid}, TB pre_resized=True')

col_covid = 'filename' if 'filename' in train_covid_df.columns else ('Image Index' if 'Image Index' in train_covid_df.columns else 'image_id')
col_tb = 'filename' if 'filename' in train_tb_df.columns else ('Image Index' if 'Image Index' in train_tb_df.columns else 'image_id')

ds_train_nih = ChestXrayDataset(df=train_nih_df, image_dir=nih_dir, image_col='Image Index', label_col='Finding Labels', task_id=0, num_classes=14, transforms=train_tf, pre_resized=pre_resized_nih, ignore_missing=True)
ds_train_covid = ChestXrayDataset(df=train_covid_df, image_dir=covid_dir, image_col=col_covid, label_col='label', task_id=2, num_classes=1, transforms=train_tf, pre_resized=pre_resized_covid, ignore_missing=True)
ds_train_tb = ChestXrayDataset(df=train_tb_df, image_dir=tb_dir, image_col=col_tb, label_col='label', task_id=1, num_classes=1, transforms=train_tf, pre_resized=True, ignore_missing=True)

# Gan file_map vao dataset de tang toc truy xuat
if file_map_nih: ds_train_nih.file_map.update(file_map_nih)
if file_map_covid: ds_train_covid.file_map.update(file_map_covid)
if file_map_tb: ds_train_tb.file_map.update(file_map_tb)

concat_train = ConcatDataset([ds_train_nih, ds_train_covid, ds_train_tb])

offset_nih = 0
offset_covid = len(ds_train_nih)
offset_tb = offset_covid + len(ds_train_covid)

nih_indices = list(range(offset_nih, offset_covid))
covid_indices = list(range(offset_covid, offset_tb))
tb_indices = list(range(offset_tb, len(concat_train)))

BATCH_SIZE = 16
batch_sampler = WeightedMultiTaskBatchSampler(
    nih_indices=nih_indices,
    covid_indices=covid_indices,
    tb_indices=tb_indices,
    batch_size=BATCH_SIZE,
    ratios=(0.60, 0.25, 0.15),
    shuffle=True,
    seed=42
)

train_loader = DataLoader(
    concat_train,
    batch_sampler=batch_sampler,
    collate_fn=multitask_collate_fn,
    num_workers=4 if os.name != 'nt' else 0,
    pin_memory=torch.cuda.is_available()
)

# Validation Loaders
ds_val_nih = ChestXrayDataset(df=val_nih_df, image_dir=nih_dir, image_col='Image Index', label_col='Finding Labels', task_id=0, num_classes=14, transforms=val_tf, pre_resized=pre_resized_nih, ignore_missing=True)
ds_val_tb = ChestXrayDataset(df=val_tb_df, image_dir=tb_dir, image_col=col_tb, label_col='label', task_id=1, num_classes=1, transforms=val_tf, pre_resized=True, ignore_missing=True)
ds_val_covid = ChestXrayDataset(df=val_covid_df, image_dir=covid_dir, image_col=col_covid, label_col='label', task_id=2, num_classes=1, transforms=val_tf, pre_resized=pre_resized_covid, ignore_missing=True)

if file_map_nih: ds_val_nih.file_map.update(file_map_nih)
if file_map_tb: ds_val_tb.file_map.update(file_map_tb)
if file_map_covid: ds_val_covid.file_map.update(file_map_covid)

val_loader_a = DataLoader(ds_val_nih, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)
val_loader_b = DataLoader(ds_val_tb, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)
val_loader_c = DataLoader(ds_val_covid, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)

print(f'[*] Unified TrainLoader: {len(train_loader)} batches/epoch (Batch Size={BATCH_SIZE}).')
print(f'[*] Val Loaders: NIH={len(val_loader_a)} batches, TB={len(val_loader_b)} batches, COVID={len(val_loader_c)} batches.')



### 5. MultiTaskChestXrayNet Architecture & Masked Loss Engine
Khởi tạo mô hình DenseNet-121 dùng chung với 3 Classification Heads lâm sàng độc lập và bộ máy Masked Multi-Task Loss cô lập gradient tuyệt đối.


In [ ]:
from src.models.multi_task_net import MultiTaskChestXrayNet
from src.models.losses import MaskedMultiTaskLoss

model = MultiTaskChestXrayNet(
    backbone_name='densenet121',
    pretrained=True,
    img_size=512,
    dropout_rate=0.2,
    num_classes_a=14,
    num_classes_b=1,
    num_classes_c=1
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[*] MultiTaskChestXrayNet: Total Params={total_params:,d} | Trainable={trainable_params:,d}')

# Positive Class Weights cho 14 benh NIH
pos_counts = [11559, 2776, 13317, 19894, 5782, 6331, 1431, 5302, 4667, 2303, 2516, 1686, 3385, 227]
total_nih = 112120
pos_weights = torch.tensor([min(50.0, (total_nih - p) / max(1, p)) for p in pos_counts], dtype=torch.float32)

criterion = MaskedMultiTaskLoss(
    weights={'head_a': 1.0, 'head_b': 0.8, 'head_c': 0.8},
    head_a_criterion='bce',
    head_c_criterion='bce',
    pos_weight_a=pos_weights
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-6)
print('[*] MaskedMultiTaskLoss & AdamW Optimizer & CosineAnnealingLR initialized.')



### 6. Multi-Task Training Loop with Mixed Precision (AMP FP16)
Thực thi vòng lặp 25 epochs huấn luyện đa nhiệm với PyTorch AMP, Cosine Annealing LR và Early Stopping.


In [ ]:
from src.models.trainer import MultiTaskTrainer

trainer = MultiTaskTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    lr_scheduler=scheduler,
    device=device,
    mixed_precision=True,
    early_stopping_patience=5,
    checkpoint_dir=output_dir,
    results_dir=results_dir,
    metric_weights={'auc_a': 0.40, 'auc_b': 0.30, 'auc_c': 0.30}
)

NUM_EPOCHS = 25
print(f'[*] Starting Multi-Task Training for {NUM_EPOCHS} Epochs on {device}...')
history_df = trainer.fit(
    train_loader=train_loader,
    val_loader_a=val_loader_a,
    val_loader_b=val_loader_b,
    val_loader_c=val_loader_c,
    epochs=NUM_EPOCHS,
    save_name='checkpoint_baseline_multitask.pth'
)



### 7. Training Loss & Multi-Task ROC-AUC Convergence Curves
Vẽ trực tiếp biểu đồ đường cong hàm mất mát đa nhiệm và tiến trình tăng trưởng điểm ROC-AUC trên tập Validation qua 25 epochs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Loss curves
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Total Loss', lw=2.5, color='#1E88E5')
if 'train_loss_a' in history_df:
    axes[0].plot(history_df['epoch'], history_df['train_loss_a'], label='Head A (NIH)', ls='--', color='#43A047')
if 'train_loss_b' in history_df:
    axes[0].plot(history_df['epoch'], history_df['train_loss_b'], label='Head B (TB)', ls='--', color='#FB8C00')
if 'train_loss_c' in history_df:
    axes[0].plot(history_df['epoch'], history_df['train_loss_c'], label='Head C (COVID)', ls='--', color='#E53935')

axes[0].set_title('Multi-Task Training Loss Convergence', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss Value')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Subplot 2: Validation ROC-AUC curves
axes[1].plot(history_df['epoch'], history_df['val_combined_score'], label='Combined Score S_val', lw=2.5, color='#8E24AA')
axes[1].plot(history_df['epoch'], history_df['val_auc_a'], label='Head A Macro AUC', ls=':', color='#43A047')
axes[1].plot(history_df['epoch'], history_df['val_auc_b'], label='Head B TB AUC', ls=':', color='#FB8C00')
axes[1].plot(history_df['epoch'], history_df['val_auc_c'], label='Head C COVID Macro AUC', ls=':', color='#E53935')
axes[1].axhline(y=0.80, color='gray', linestyle='--', alpha=0.6, label='Gate 2 Target (0.80/0.90)')

axes[1].set_title('Validation Performance across Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROC-AUC Score')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()



### 8. Comprehensive Clinical Evaluation on Independent Test Sets
Đánh giá chuyên sâu lâm sàng trên tập kiểm thử độc lập (Test Sets):
- **Head A**: Bảng số liệu AUROC 14 bệnh và 14 đường cong ROC curves.
- **Head B**: Đường cong ROC, độ nhạy (Sensitivity) & độ đặc hiệu (Specificity) của bệnh Lao phổi.
- **Head C**: Ma trận nhầm lẫn chuẩn hóa (Confusion Matrix Heatmap) trên 4 lớp COVID-19.


In [ ]:
print('=== ĐÁNH GIÁ CHUYÊN SÂU TRÊN TẬP TEST ĐỘC LẬP ===')

ds_test_nih = ChestXrayDataset(test_nih_df, nih_dir, 'Finding Labels', 0, 14, val_tf, ignore_missing=True)
ds_test_tb = ChestXrayDataset(test_tb_df, tb_dir, 'label', 1, 1, val_tf, ignore_missing=True)
ds_test_covid = ChestXrayDataset(test_covid_df, covid_dir, 'label', 2, 1, val_tf, ignore_missing=True)

if file_map_nih: ds_test_nih.file_map.update(file_map_nih)
if file_map_tb: ds_test_tb.file_map.update(file_map_tb)
if file_map_covid: ds_test_covid.file_map.update(file_map_covid)

test_loader_a = DataLoader(ds_test_nih, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)
test_loader_b = DataLoader(ds_test_tb, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)
test_loader_c = DataLoader(ds_test_covid, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)

# 1. EVALUATION HEAD A (NIH ChestX-ray14)
model.eval()
all_preds_a, all_targets_a = [], []
with torch.no_grad():
    for batch in test_loader_a:
        imgs = batch['images'].to(device)
        preds = model.predict(imgs)['head_a'].cpu().numpy()
        all_preds_a.append(preds)
        all_targets_a.append(torch.stack(batch['labels']).numpy())

y_pred_a = np.concatenate(all_preds_a, axis=0)
y_true_a = np.concatenate(all_targets_a, axis=0)

nih_disease_names = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia'
]

per_disease_auc = {}
plt.figure(figsize=(10, 8))
for i, name in enumerate(nih_disease_names):
    if len(np.unique(y_true_a[:, i])) > 1:
        auc = roc_auc_score(y_true_a[:, i], y_pred_a[:, i])
        per_disease_auc[name] = auc
        fpr, tpr, _ = roc_curve(y_true_a[:, i], y_pred_a[:, i])
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
    else:
        per_disease_auc[name] = 0.50

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title('Head A: ROC Curves for 14 Thoracic Pathologies (NIH Test Set)', fontsize=13, fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

macro_auc_a = float(np.mean(list(per_disease_auc.values())))
df_auc_a = pd.DataFrame(list(per_disease_auc.items()), columns=['Pathology', 'AUC']).sort_values('AUC', ascending=False)
print(f'\n[Head A] Macro AUROC: {macro_auc_a:.4f}')
print(df_auc_a.to_string(index=False))



In [ ]:
# 2. EVALUATION HEAD B (Tuberculosis Screening)
all_preds_b, all_targets_b = [], []
with torch.no_grad():
    for batch in test_loader_b:
        imgs = batch['images'].to(device)
        preds = model.predict(imgs)['head_b'].cpu().numpy().squeeze()
        all_preds_b.append(preds)
        all_targets_b.append(torch.stack(batch['labels']).numpy().squeeze())

y_pred_b = np.concatenate(all_preds_b, axis=0)
y_true_b = np.concatenate(all_targets_b, axis=0)

if len(np.unique(y_true_b)) > 1:
    auc_b = roc_auc_score(y_true_b, y_pred_b)
    fpr_b, tpr_b, thresh_b = roc_curve(y_true_b, y_pred_b)
    
    pred_bin_b = (y_pred_b >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_b, pred_bin_b).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr_b, tpr_b, color='#FB8C00', lw=2.5, label=f'TB ROC (AUC = {auc_b:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.set_title('Head B: ROC Curve for Tuberculosis Screening', fontsize=12, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'[Head B] TB AUROC: {auc_b:.4f} | Sensitivity: {sensitivity:.4f} | Specificity: {specificity:.4f}')
else:
    auc_b = 0.50
    print('[Head B] Single class present in test batch.')



In [ ]:
# 3. EVALUATION HEAD C (COVID-19 Binary Screening - COVIDx CXR-4 Benchmark)
all_preds_c, all_targets_c = [], []
with torch.no_grad():
    for batch in test_loader_c:
        imgs = batch['images'].to(device)
        preds = model.predict(imgs)['head_c'].cpu().numpy().flatten()
        all_preds_c.extend(preds)
        targets_c = torch.stack(batch['labels']).numpy().flatten()
        all_targets_c.extend(targets_c)

y_pred_c = np.array(all_preds_c)
y_true_c = np.array(all_targets_c)

if len(np.unique(y_true_c)) > 1:
    auc_c = float(roc_auc_score(y_true_c, y_pred_c))
    fpr_c, tpr_c, thresholds_c = roc_curve(y_true_c, y_pred_c)
    youden_j_c = tpr_c - fpr_c
    opt_idx_c = np.argmax(youden_j_c)
    opt_thresh_c = float(thresholds_c[opt_idx_c])
    y_pred_labels = (y_pred_c >= opt_thresh_c).astype(int)
    
    cm = confusion_matrix(y_true_c, y_pred_labels)
    tn, fp, fn, tp = cm.ravel()
    sens_c = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec_c = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Reds',
        xticklabels=['Normal', 'COVID-19'],
        yticklabels=['Normal', 'COVID-19']
    )
    plt.title(f'Head C: Confusion Matrix (COVIDx CXR-4 Binary | tau*={opt_thresh_c:.2f})', fontsize=11, fontweight='bold')
    plt.xlabel('Predicted Class')
    plt.ylabel('Ground Truth Class')
    plt.tight_layout()
    plt.show()

    print(f'[Head C - COVID-19] AUROC: {auc_c:.4f} | Optimal Threshold tau*: {opt_thresh_c:.4f}')
    print(f'[Head C - COVID-19] Sensitivity: {sens_c*100:.2f}% | Specificity: {spec_c*100:.2f}%')
    print('\nClassification Report:')
    print(classification_report(y_true_c, y_pred_labels, target_names=['Normal', 'COVID-19']))
else:
    auc_c = 0.50
    opt_thresh_c = 0.50
    sens_c = 0.0
    spec_c = 0.0


### 9. Gate 2 Verification Scorecard & Artifact Export
Đối chiếu các chỉ số nghiệm thu Gate 2 và xuất toàn bộ file số liệu CSV ra `results/baseline/`.


In [ ]:
os.makedirs(results_dir, exist_ok=True)

# Export CSV tables
df_auc_a.to_csv(os.path.join(results_dir, 'per_disease_auc.csv'), index=False)
pd.DataFrame([{'Macro_AUC': macro_auc_a}]).to_csv(os.path.join(results_dir, 'head_a_macro.csv'), index=False)
pd.DataFrame([{'AUROC': auc_b, 'Sensitivity': sensitivity if 'sensitivity' in locals() else 0, 'Specificity': specificity if 'specificity' in locals() else 0}]).to_csv(os.path.join(results_dir, 'tb_metrics.csv'), index=False)
pd.DataFrame([{'Macro_AUROC': auc_c, 'Macro_F1': macro_f1_c if 'macro_f1_c' in locals() else 0}]).to_csv(os.path.join(results_dir, 'covid_metrics.csv'), index=False)

print('=== DELIVERY GATE 2 COMPLIANCE SCORECARD ===')
print(f' 1. Head A (NIH 14 Pathologies Macro AUC): {macro_auc_a:.4f}  (Target: >= 0.80)  -> {"PASS" if macro_auc_a >= 0.80 else "PENDING CLOUD RUN"}')
print(f' 2. Head B (TB Screening AUC)            : {auc_b:.4f}  (Target: >= 0.90)  -> {"PASS" if auc_b >= 0.90 else "PENDING CLOUD RUN"}')
print(f' 3. Head C (COVIDx CXR-4 Binary Screening AUC)  : {auc_c:.4f}  (Target: >= 0.90)  -> {"PASS" if auc_c >= 0.90 else "PENDING CLOUD RUN"}')
print(f'\nArtifacts successfully exported to: {results_dir}')
print(f'Checkpoint location: {os.path.join(output_dir, "checkpoint_baseline_multitask.pth")}')

